# Hafta 11 — GPT-2 eğitimini hızlandırma

Bu defter **kendi hafta 10 modelimizi** kullanır; pretrained ağırlık indirmez.
Runtime → Change runtime type → T4/A100 GPU seçin ve hücreleri sırayla çalıştırın.
Kaynaklar deftere gömülüdür; GitHub push, Drive bağlama veya özel corpus gerekmez.

Hızlandırma sırası: FP32 → TF32 → BF16/FP16 → compile → zorlanan FlashAttention → 50.304 vocab.
Desteklenmeyen veya hata veren aşama gerekçesiyle kaydedilir, sonraki ayara eklenmez.
Ölçümde batch/context sabittir. Warmup/derleme ölçüm dışında, ayrıca kayıttadır.
GPU adımları birkaç dakika, ilk derleme daha uzun sürebilir. Sayısal GPU sonuçları çalıştırılana kadar yoktur.

Demo metni yalnız eğitim hattı içindir; büyük veri/genelleme değerlendirmesi gelecek haftadır.


## 1. Kaynakları doğrula ve bağımlılıkları kur

In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path

ROOT = Path('/content/neuron-to-gpt-week11')
ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_HASHES = {'week10/model.py': 'c3880320d61666d7c79f861dba74fb242ad09085fb7e03dd45ea64dde55f9535', 'week11/training.py': '4f2726530f52a6b2a4ba42afde9d5a2b1848445aa94c315f2c7d3aba2051fe05', 'week11/run_experiments.py': '6510737725b9effd2bfdf98c05e02e5e728725061ceef6158a55e0bf131f1b74', 'week11/test_training.py': 'c9a9b7ce290399f0ce9ec883b8edcdaf7f6835da92a48401ae54191cfefdffb7', 'week11/requirements.txt': 'ecd070dc12616b0a12ab23ebc74c6dde9e7a93091bcfef061aff27032249e2be', 'week11/data/demo.txt': '06d8f14166ad929c8be6a9179cf4dda3b73a8981562f4e25d6cc854597ad8b79'}
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'aee44e99678a4a42e338bad17a3689465f0e96a2703d5cc13e1008d3dd9c77b9'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert set(archive.namelist()) == set(SOURCE_HASHES)
    for filename, expected in SOURCE_HASHES.items():
        raw = archive.read(filename)
        assert hashlib.sha256(raw).hexdigest() == expected
        destination = ROOT / filename
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(raw)
os.environ['TIKTOKEN_CACHE_DIR'] = str(ROOT / 'week11/.cache/tiktoken')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'week11/requirements.txt')], check=True)
print('Sources verified:', json.dumps(SOURCE_HASHES, indent=2))


## 2. GPU ve deney ayarları
Bellek yetmezse batch/context'i burada küçültüp **tüm hız aşamalarını aynı ayarla** yeniden çalıştırın.

In [ ]:
import re, torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU, ardından yeniden bağlanın.'
GPU_NAME = torch.cuda.get_device_name(0)
RUN_NAME = re.sub(r'[^a-z0-9]+', '_', GPU_NAME.lower()).strip('_')
RESULTS = ROOT / 'week11/results' / RUN_NAME
RESULTS.mkdir(parents=True, exist_ok=True)
BATCH_SIZE, SEQUENCE_LENGTH = 2, 128
STEPS, LR_WARMUP, ACCUMULATION = 300, 30, 4
print('GPU:', GPU_NAME, '| torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('T4: TF32 atlanır; FP16+GradScaler kullanılır. A100: TF32/BF16 desteklenir.')
print('FlashAttention yalnız zorlanan flash backend gerçekten çalışırsa ölçülür.')
subprocess.run(['nvidia-smi'], check=True)

def run_task(task):
    subprocess.run([
        sys.executable, '-u', 'week11/run_experiments.py', '--task', task,
        '--device', 'cuda', '--output-dir', str(RESULTS),
        '--batch-size', str(BATCH_SIZE), '--sequence-length', str(SEQUENCE_LENGTH),
        '--steps', str(STEPS), '--lr-warmup', str(LR_WARMUP),
        '--accumulation-steps', str(ACCUMULATION),
    ], cwd=ROOT, check=True)


## 3. Loader, accumulation, optimizer, schedule ve başlangıç kontrolleri

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'week11', '-p', 'test_*.py'], cwd=ROOT, check=True)


## 4. Birikimli GPU hız ölçümleri
Her aşamada 5 warmup ve 20 ölçülen güncelleme; ham süreler JSON'a yazılır. Flash backend desteklenmezse hata nedeni tabloda görünür.

In [ ]:
run_task('benchmark')


## 5. Tam demo metninde 300 güncelleme
AdamW β=(0.9,0.95), clipping=1, decay=0.1; 30 adım warmup ve cosine.
Etkin batch: 2×128×4=1.024 token. Videodaki 524.288 token için accumulation=2.048 gerekir; kısa ödev koşusunda bu maliyetli hedef kullanılmaz.
FP16 atlanan güncellemeleri ayrıca raporlanır. Her satırda loss, norm, LR, süre ve token/s vardır.

In [ ]:
run_task('train')


## 6. Ölçülen tablo ve LR grafiği
T4'te en az iki iyileştirme ölçülmesini bekliyoruz: FP16, compile ve vocab padding adayları. Sonuç raporu şartın sağlanıp sağlanmadığını sayar.

In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((RESULTS / 'submission_report.md').read_text(encoding='utf-8')))
display(Image(filename=str(RESULTS / 'learning_rate.png')))


## 7. Kanıt dosyalarını indir
Başka GPU'ya geçmeden ZIP'i kaydedin. Özel bir metin kullanırsanız kaynak metin ZIP'e eklenmez; adı ve hash'i kaydedilir.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(ROOT / ('week11_' + RUN_NAME + '_results')), 'zip', RESULTS)
files.download(archive)
print('ZIP dosyasını cihaz değiştirmeden önce kaydedin:', archive)


## Açıklamalar ve kaynaklar

Weight tying, 50.257×768=38.597.376 parametreyi paylaşır. Residual dallarda
std=0.02/√(2L), art arda toplanan katkıların başlangıç varyansını sınırlar.
K eşit büyüklükte mikro batch için loss/K, gradient toplamını büyük batch'in
ortalama gradient'ine dönüştürür. Clipping accumulation sonunda bir kez uygulanır.

GPT-3 makalesindeki token bütçeli warmup/decay ve batch ramp-up'ın tamamını
yeniden üretmiyoruz; videodaki kısa deney için update sayısına uyarlıyoruz.
Ek DDP/Türkçe tokenizer eğitimi bu defterde yapılmıyor.

- [Karpathy videosu 1:02:00–2:46:52](https://www.youtube.com/watch?v=l8pRSuU81PU&t=3720s)
- [GPT-3, Appendix B](https://arxiv.org/html/2005.14165v4#A2)
- [PyTorch AMP accumulation](https://docs.pytorch.org/docs/2.14/notes/amp_examples.html#gradient-accumulation)
- [PyTorch SDPA backend seçimi](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.scaled_dot_product_attention.html)
